In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [2]:
# LOAD DATASET
df = pd.read_csv("Shark Tank US dataset.csv")

In [3]:
df.columns

Index(['Season Number', 'Startup Name', 'Episode Number', 'Pitch Number',
       'Season Start', 'Season End', 'Original Air Date', 'Industry',
       'Business Description', 'Company Website', 'Pitchers Gender',
       'Pitchers Average Age', 'Pitchers City', 'Pitchers State',
       'Entrepreneur Names', 'Multiple Entrepreneurs', 'US Viewership',
       'Original Ask Amount', 'Original Offered Equity', 'Valuation Requested',
       'Got Deal', 'Total Deal Amount', 'Total Deal Equity', 'Deal Valuation',
       'Number of Sharks in Deal', 'Investment Amount Per Shark',
       'Equity Per Shark', 'Royalty Deal', 'Advisory Shares Equity', 'Loan',
       'Deal Has Conditions', 'Barbara Corcoran Investment Amount',
       'Barbara Corcoran Investment Equity', 'Mark Cuban Investment Amount',
       'Mark Cuban Investment Equity', 'Lori Greiner Investment Amount',
       'Lori Greiner Investment Equity', 'Robert Herjavec Investment Amount',
       'Robert Herjavec Investment Equity', 'Daymon

In [4]:
# Clean target: drop missing pitches and keep only 1 / 0
df = df.dropna(subset=["Got Deal"]).copy()
df["Got Deal"] = df["Got Deal"].astype(int)

In [5]:
# Drop Post-Pitch Leakage (events decided after pitch ends)
leakage_cols = [
    "Total Deal Amount", "Total Deal Equity", "Deal Valuation",
    "Number of Sharks in Deal", "Investment Amount Per Shark",
    "Equity Per Shark", "Royalty Deal", "Advisory Shares Equity",
    "Loan", "Deal Has Conditions", "Guest Name"
]

In [6]:
df = df.drop(columns=[c for c in leakage_cols if c in df.columns], errors="ignore")

In [7]:
numerical_cols = [
    "Original Ask Amount", 
    "Original Offered Equity", 
    "Valuation Requested", 
    "US Viewership"
]

In [8]:
categorical_cols = [
    "Industry", 
    "Pitchers Gender"
]

In [9]:
X = df[numerical_cols + categorical_cols].copy()
y = df["Got Deal"]

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.20, random_state=42, stratify=y)

## Feature Engineering Pipelines

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [12]:
# Numerical Pipeline

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [13]:
# Categorical pipeline

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [14]:
preprocessor = ColumnTransformer([
    ('num_pipeline', num_pipeline, numerical_cols),
    ('cat_pipeline', cat_pipeline, categorical_cols)
])

In [15]:
# Fit on training data and transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## Model Comparison

In [16]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [18]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        eval_metric='logloss',
        random_state=42
    )
}

In [19]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluate_model(X_train, y_train, X_test, y_test, models):
  report = []

  for name, model in models.items():
    # 1. Fit the model
    model.fit(X_train, y_train)

    # 2. Predict classes and probabilities
    y_pred = model.predict(X_test)
    y_prob = (
        model.predict_proba(X_test)[:, 1]
        if hasattr(model, "predict_proba")
        else None
    )

    # 3. Compute key classification metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    report.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc, 4),
    })

  report_df = pd.DataFrame(report)

  # 4. Determine the best model using a Composite Rank across all metrics:
  # Ranks each metric descending (1 = best score). Lower mean rank = better overall.
  eval_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
  report_df['Composite_Rank'] = (
      report_df[eval_metrics].rank(ascending=False, method='min').mean(axis=1)
  )

  # Sort so the overall strongest model is on top
  report_df = report_df.sort_values(
      by='Composite_Rank', ascending=True
  ).reset_index(drop=True)

  # 5. Extract best model details
  best_model_name = report_df.iloc[0]['Model']
  best_model = models[best_model_name]

  return report_df, best_model_name, best_model

In [20]:
report_df, best_model_name, best_model = evaluate_model(
    X_train_processed, y_train, X_test_processed, y_test, models
)

In [21]:
print("=== Model Performance Comparison ===")
print(report_df.to_string(index=False))

=== Model Performance Comparison ===
              Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Composite_Rank
Logistic Regression    0.6296     0.6409  0.9071    0.7511   0.5967             1.4
      Random Forest    0.6263     0.6333  0.9344    0.7550   0.5795             2.0
      Decision Tree    0.6027     0.6383  0.8197    0.7177   0.5655             3.0
            XGBoost    0.5960     0.6400  0.7869    0.7059   0.5543             3.6


In [22]:
print(f"\n Best Overall Model: {best_model_name}")


 Best Overall Model: Logistic Regression


In [23]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = best_model.predict(X_test_processed)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 21  93]
 [ 17 166]]
              precision    recall  f1-score   support

           0       0.55      0.18      0.28       114
           1       0.64      0.91      0.75       183

    accuracy                           0.63       297
   macro avg       0.60      0.55      0.51       297
weighted avg       0.61      0.63      0.57       297



In [24]:
# Save Artifacts for Streamlit

joblib.dump(best_model, "best_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")
print(" Saved 'best_model.pkl' and 'preprocessor.pkl' successfully!")

 Saved 'best_model.pkl' and 'preprocessor.pkl' successfully!


## Deployment (Streamlit)

In [25]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Load artifacts saved in cell [34]
model = joblib.load("best_model.pkl")
preprocessor = joblib.load("preprocessor.pkl")

st.set_page_config(page_title="Shark Tank Predictor", layout="centered")
st.title("🦈 Shark Tank Deal Predictor")
st.write("Enter pitch details to predict whether the startup gets an investment offer.")

# Input fields
ask = st.number_input("Original Ask Amount ($)", min_value=1000, value=100000, step=5000)
equity = st.number_input("Original Offered Equity (%)", min_value=0.5, max_value=100.0, value=10.0, step=0.5)
viewers = st.number_input("US Viewership (Millions)", min_value=0.1, value=5.5, step=0.1)

# Implied valuation calculation
valuation = (ask / (equity / 100.0)) if equity > 0 else 0
st.caption(f"Implied Valuation: **${valuation:,.2f}**")

industry = st.selectbox("Industry", [
    "Food and Beverage", "Fashion / Beauty", "Lifestyle / Home", 
    "Children / Education", "Fitness / Health", "Software / Tech", "Other"
])
gender = st.selectbox("Pitchers Gender", ["Male", "Female", "Mixed Team"])

if st.button("Predict Deal"):
    # Build dataframe with exact training feature names
    input_data = pd.DataFrame([{
        "Original Ask Amount": ask,
        "Original Offered Equity": equity,
        "Valuation Requested": valuation,
        "US Viewership": viewers,
        "Industry": industry,
        "Pitchers Gender": gender
    }])
    
    # Preprocess & predict
    input_proc = preprocessor.transform(input_data)
    pred = model.predict(input_proc)[0]
    prob = model.predict_proba(input_proc)[0][1]
    
    st.divider()
    if pred == 1:
        st.success(f"🎉 **Likely to get a Deal!** (Confidence: {prob * 100:.1f}%)")
    else:
        st.error(f"❌ **Unlikely to get a Deal.** (Confidence: {(1 - prob) * 100:.1f}%)")

Writing app.py


In [ ]:
%pip install streamlit

In [ ]:
!python -m streamlit run app.py
# or run --->   streamlit run app.py  in terminal 

In [28]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn
joblib

Writing requirements.txt
